### Notebook for processing the validation of the Futhark results

In [ ]:
import ast
import re
import numpy as np
import pandas as pd
from pathlib import Path

##### Functions for processing Matlab and Futhark Prices

In [6]:
FUT_TYPE_SUFFIX = re.compile(r'(f16|f32|f64|i8|i16|i32|i64|u8|u16|u32|u64)\b')

def parse_futhark_prices(path):
    """Parse the price matrix from line 1 of a Futhark validate_solve .val file.

    The first line is a Futhark literal of shape [c][Ax], e.g.
    `[[200.0f64, 169.6f64, ...], [260.0f64, ...], ...]`.
    Returns a numpy.ndarray of shape (c, Ax) with dtype float64.
    """
    with open(path) as f:
        first_line = f.readline()
    cleaned = FUT_TYPE_SUFFIX.sub('', first_line)
    nested = ast.literal_eval(cleaned)
    return np.asarray(nested, dtype=np.float64)

def parse_matlab_prices(path):
    """Parse the price matrix from a MATLAB validate_run_illustrations_test .dat file.

    The file is a CSV (one row per car type, columns = ages 0..Ax-1).
    Returns a numpy.ndarray of shape (c, Ax) with dtype float64.
    """
    return np.loadtxt(path, delimiter=',', dtype=np.float64)

In [7]:
parse_futhark_prices('validation_files/run_equilibrium-local-validate_solve-2-3-25-5-0-M.val')

array([[200.        , 169.2948619 , 141.47740163, 117.0506919 ,
         95.96637377,  77.96530383,  62.62372555,  49.514447  ,
         38.31752597,  28.83231542,  20.94596407,  14.59768623,
          9.74911362,   6.33721219,   4.18410726,   2.96706019,
          2.33557   ,   2.0274529 ,   1.88412023,   1.81988041,
          1.79051281,   1.7704009 ,   1.73140671,   1.5988312 ,
          1.06290795],
       [260.        , 225.25506523, 193.02633663, 163.79794474,
        137.68705753, 114.73608295,  94.842273  ,  77.72158179,
         62.98567507,  50.27381226,  39.32270211,  29.96686878,
         22.11232877,  15.70826562,  10.72367414,   7.11425982,
          4.74873215,   3.35688607,   2.60697298,   2.22655565,
          2.03958087,   1.94245189,   1.8640367 ,   1.70520958,
          1.1400604 ],
       [260.        , 225.25506523, 193.02633663, 163.79794474,
        137.68705753, 114.73608295,  94.842273  ,  77.72158179,
         62.98567507,  50.27381226,  39.32270211,  29.9668

In [8]:
parse_matlab_prices('matlab_results_for_validation/matlab-validate_solve-2-3-25-5-0.dat')

array([[200.        , 169.29486084, 141.4773999 , 117.05068984,
         95.96637165,  77.96530179,  62.62372362,  49.5144452 ,
         38.3175243 ,  28.83231391,  20.94596275,  14.59768512,
          9.7491127 ,   6.33721146,   4.18410672,   2.96705985,
          2.3355698 ,   2.02745279,   1.88412018,   1.81988038,
          1.7905128 ,   1.7704009 ,   1.73140671,   1.5988312 ,
          1.06290794],
       [260.        , 225.25506359, 193.0263338 , 163.79794114,
        137.68705352, 114.73607879,  94.84226882,  77.72157762,
         62.98567095,  50.27380827,  39.32269838,  29.96686546,
         22.11232603,  15.70826358,  10.72367279,   7.11425896,
          4.7487316 ,   3.35688573,   2.60697279,   2.22655554,
          2.03958082,   1.94245186,   1.86403669,   1.70520957,
          1.14006039],
       [260.        , 225.25506359, 193.0263338 , 163.79794114,
        137.68705352, 114.73607879,  94.84226882,  77.72157762,
         62.98567095,  50.27380827,  39.32269838,  29.9668

##### Functions for comparing Futhark and Matlab

In [9]:
_FUT_NAME_RE = re.compile(r'validate_solve-(\d+)-(\d+)-(\d+)-(\d+)-(\d+)-[A-Z]\.val$')

def compare_validation(futhark_name, rtol=1e-4, atol=1e-6,
                         val_dir='validation_files',
                         mat_dir='matlab_results_for_validation'):
      """Compare prices in a Futhark .val file against the matching MATLAB .dat file.

      Pass/fail uses numpy.allclose semantics:
          |fut - mat| <= atol + rtol * |mat|

      Parameters
      ----------
      futhark_name : str
          Filename inside `val_dir`, e.g.
          'run_equilibrium-local-validate_solve-2-7-25-5-0-C.val'.
      rtol : float, default 1e-4
          Relative tolerance (dominates for large prices).
      atol : float, default 1e-6
          Absolute tolerance floor (dominates near scrap, where prices ~ 0).
      val_dir, mat_dir : str or Path
          Directories holding the Futhark and MATLAB validation files.

      Returns
      -------
      None
          If no MATLAB file matches the parameter tuple in `futhark_name`.
      dict
          Keys:
            'within_tol'   bool   passes np.allclose(fut, mat, rtol, atol)
            'max_abs_diff' float  max |fut - mat|
            'max_rel_diff' float  max |fut - mat| / max(|mat|, atol)
            'matlab_path'  Path
      """
      m = _FUT_NAME_RE.search(futhark_name)
      if m is None:
          raise ValueError(f"Could not extract parameter tuple from: {futhark_name}")
      n, c, abar, acc0, trans = m.groups()

      matlab_name = f"matlab-validate_solve-{n}-{c}-{abar}-{acc0}-{trans}.dat"
      matlab_path = Path(mat_dir) / matlab_name
      if not matlab_path.exists():
          return None

      fut = parse_futhark_prices(Path(val_dir) / futhark_name)
      mat = parse_matlab_prices(matlab_path)

      abs_diff = np.abs(fut - mat)
      max_abs_diff = float(np.max(abs_diff))
      max_rel_diff = float(np.max(abs_diff / np.maximum(np.abs(mat), atol)))
      return {
          'within_tol':   bool(np.allclose(fut, mat, rtol=rtol, atol=atol)),
          'max_abs_diff': max_abs_diff,
          'max_rel_diff': max_rel_diff,
          'matlab_path':  matlab_path,
      }

##### Futhark Functions for parsing and summarizing validation files

In [10]:
def parse_futhark_validation(path):
      """Parse all fields from a Futhark validate_solve .val file.

      Returns a dict with:
        'prices'        ndarray  shape (c, Ax), float64
        'max_abs_ed'    float
        'stat_res'      float
        'norm_err'      float
        'min_q'         float
        'iter'          int      Newton outer iterations
        'conv'          bool     convergence flag
        'sa_iters_tot'  list[int]  per-household SA iteration totals
        'nk_iters_tot'  list[int]  per-household NK iteration totals
        'rtrips_tot'    list[int]  per-household round-trip totals
      """
      with open(path) as f:
          lines = f.read().splitlines()

      def strip(s):
          return FUT_TYPE_SUFFIX.sub('', s)

      return {
          'prices':       np.asarray(ast.literal_eval(strip(lines[0])), dtype=np.float64),
          'max_abs_ed':   float(ast.literal_eval(strip(lines[1]))),
          'stat_res':     float(ast.literal_eval(strip(lines[2]))),
          'norm_err':     float(ast.literal_eval(strip(lines[3]))),
          'min_q':        float(ast.literal_eval(strip(lines[4]))),
          'iter':         int(ast.literal_eval(strip(lines[5]))),
          'conv':         lines[6].strip() == 'true',
          'sa_iters_tot': list(ast.literal_eval(strip(lines[7]))),
          'nk_iters_tot': list(ast.literal_eval(strip(lines[8]))),
          'rtrips_tot':   list(ast.literal_eval(strip(lines[9]))),
      }

In [11]:
def summarize_futhark_run(futhark_name, rtol=1e-4, atol=1e-6,
                            val_dir='validation_files',
                            mat_dir='matlab_results_for_validation'):
      """All Futhark validation fields (except the price matrix) plus the
      MATLAB comparison verdict.

      See parse_futhark_validation for the remaining field names. Adds:
        'within_tol'    bool or None  (None if no matching MATLAB file)
        'max_abs_diff'  float or None
        'max_rel_diff'  float or None
        'matlab_path'   Path  or None
      """
      fields = parse_futhark_validation(Path(val_dir) / futhark_name)
      del fields['prices']
      cmp = compare_validation(futhark_name, rtol=rtol, atol=atol,
                               val_dir=val_dir, mat_dir=mat_dir)
      if cmp is None:
          fields.update(within_tol=None, max_abs_diff=None,
                        max_rel_diff=None, matlab_path=None)
      else:
          fields.update(cmp)
      return fields

##### Summarize all Futhark Validation Files

In [12]:
_FUT_FILE_RE = re.compile(
      r'^(?P<run_equi>[^-]+)-(?P<val_name>.+?)-validate_solve-'
      r'(?P<n>\d+)-(?P<c>\d+)-(?P<abar>\d+)-(?P<acc0>\d+)-(?P<trans>\d+)-'
      r'(?P<backend>[A-Z])\.val$'
  )

def summarize_all_futhark(rtol=1e-4, atol=1e-6,
                            val_dir='validation_files',
                            mat_dir='matlab_results_for_validation'):
      """Summarize every .val file in `val_dir`.

      Each row is a dict carrying:
        filename       str       the .val filename
        run_equi       str       e.g. 'run_equilibrium' / 'run_equilibrium_man'
        val_name       str       VAL_NAME tag, e.g. 'local' / 'default'
        n, c, abar, acc0, trans  int   parameter tuple
        backend        str       'C', 'M', 'O', 'U'
      plus everything `summarize_futhark_run` returns (prices, max_abs_ed,
      stat_res, norm_err, min_q, iter, conv, sa_iters_tot, nk_iters_tot,
      rtrips_tot, within_tol, max_abs_diff, max_rel_diff, matlab_path).

      Files whose names don't match the validate_solve pattern are skipped.
      """
      val_dir = Path(val_dir)
      int_keys = {'n', 'c', 'abar', 'acc0', 'trans'}
      rows = []
      for path in sorted(val_dir.glob('*.val')):
          m = _FUT_FILE_RE.match(path.name)
          if m is None:
              continue
          meta = {k: int(v) if k in int_keys else v
                  for k, v in m.groupdict().items()}
          meta['filename'] = path.name
          summary = summarize_futhark_run(path.name, rtol=rtol, atol=atol,
                                          val_dir=val_dir, mat_dir=mat_dir)
          rows.append({**meta, **summary})
      return rows

In [13]:
rows = summarize_all_futhark()
print(rows[0])  # Print the first row, adjust index as needed

{'run_equi': 'run_equilibrium', 'val_name': 'local', 'n': 2, 'c': 1, 'abar': 25, 'acc0': 5, 'trans': 0, 'backend': 'C', 'filename': 'run_equilibrium-local-validate_solve-2-1-25-5-0-C.val', 'max_abs_ed': 1.77606e-10, 'stat_res': 3e-15, 'norm_err': 0.0, 'min_q': 0.00047250047593, 'iter': 7, 'conv': True, 'sa_iters_tot': [30, 33], 'nk_iters_tot': [12, 7], 'rtrips_tot': [7, 7], 'within_tol': True, 'max_abs_diff': 2.064203300733425e-06, 'max_rel_diff': 1.0413759811939476e-07, 'matlab_path': WindowsPath('matlab_results_for_validation/matlab-validate_solve-2-1-25-5-0.dat')}


##### Benchmark parsing

In [14]:
def parse_matlab_benchmark(path):
    """Parse matlab_eqb_<variant>.dat. Columns: n c abar acc0 trans mean stdev se."""
    arr = np.loadtxt(path)
    if arr.ndim == 1:
        arr = arr[None, :]
    cols = ['n', 'c', 'abar', 'acc0', 'trans', 'mean', 'stdev', 'se']
    return [dict(zip(cols, row)) for row in arr]

def parse_futhark_benchmark(path):
    """Parse a Futhark bench_solve_<variant>.dat (or saved copy).
    Columns: n c abar acc0 trans backend mean stdev se."""
    rows = []
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) != 9:
                continue
            rows.append({
                'n':       int(parts[0]),  'c':       int(parts[1]),
                'abar':    int(parts[2]),  'acc0':    int(parts[3]),
                'trans':   int(parts[4]),  'backend': parts[5],
                'mean':    float(parts[6]),
                'stdev':   float(parts[7]),
                'se':      float(parts[8]),
            })
    return rows

In [15]:
_SAVED_BENCH_RE = re.compile(
      r'^(?P<run_equi>[^-]+)-(?P<val_name>.+?)-bench(?P<runs>\d+)-'
      r'(?P<variant>cars|households|age)\.dat$'
  )

def bench_time_table(variant, saved_dir='saved_futhark_benchmarks'):
    """Long-format DataFrame of bench mean & stdev for the given variant.
    One row per (source, run_equi, val_name, backend, parameter tuple)."""
    rows = []

    mat_path = Path(f'matlab_eqb_{variant}.dat')
    if mat_path.exists():
        for r in parse_matlab_benchmark(mat_path):
            rows.append({
                'source': 'matlab', 'run_equi': '-', 'val_name': '-',
                'backend': '-',     'runs': '-',
                'n': int(r['n']), 'c': int(r['c']), 'abar': int(r['abar']),
                'mean': r['mean'], 'stdev': r['stdev'],
            })

    for path in sorted(Path(saved_dir).glob(f'*-{variant}.dat')):
        m = _SAVED_BENCH_RE.match(path.name)
        if m is None:
            continue
        meta = m.groupdict()
        for r in parse_futhark_benchmark(path):
            rows.append({
                'source':   'futhark',
                'run_equi': meta['run_equi'],
                'val_name': meta['val_name'],
                'backend':  r['backend'],
                'runs':     int(meta['runs']),
                'n': r['n'], 'c': r['c'], 'abar': r['abar'],
                'mean': r['mean'], 'stdev': r['stdev'],
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        sort_key = {'cars': 'c', 'households': 'n', 'age': 'abar'}[variant]
        df = df.sort_values([sort_key, 'source', 'run_equi', 'val_name', 'backend'])
    return df.reset_index(drop=True)

##### Comparison Table for Validation

In [16]:
def comparison_table(variant, rtol=1e-4, atol=1e-6,
                       val_dir='validation_files',
                       mat_dir='matlab_results_for_validation'):
      """DataFrame summarising every .val file that belongs to this variant.

      Each row: one (run_equi, val_name, backend, parameter tuple) combination,
      with columns including within_tol, max_abs_diff, max_rel_diff, and the
      parsed Futhark fields (iter, conv, sa_iters_tot, nk_iters_tot, etc.).
      """
      rows = summarize_all_futhark(rtol=rtol, atol=atol,
                                   val_dir=val_dir, mat_dir=mat_dir)
      if variant == 'cars':
          keep = lambda r: r['n'] == 2 and r['abar'] == 25
          sort_key = 'c'
      elif variant == 'households':
          keep = lambda r: r['c'] == 7 and r['abar'] == 25
          sort_key = 'n'
      elif variant == 'age':
          keep = lambda r: r['n'] == 2 and r['c'] == 7
          sort_key = 'abar'
      else:
          raise ValueError(f"Unknown variant: {variant!r}")

      df = pd.DataFrame([r for r in rows if keep(r)])
      if not df.empty:
          df = (df.drop(columns=['matlab_path'], errors='ignore')
                  .sort_values(['run_equi', 'val_name', 'backend', sort_key])
                  .reset_index(drop=True))
      return df

##### Tables

In [17]:
for v in ['cars', 'households', 'age']:
    print(f"\n=== {v}: bench times ===")
    display(bench_time_table(v))

for v in ['cars', 'households', 'age']:
    print(f"\n=== {v}: MATLAB comparison + Futhark diagnostics ===")
    display(comparison_table(v))


=== cars: bench times ===


NameError: name 'pd' is not defined